# Build plot-ready and model-ready tables

AnnData is the right home for aligned single-cell data, but most plotting and statistical APIs consume pandas DataFrames. This notebook shows three deliberate extraction boundaries: cell-level long data for faceted plots, group-level summaries for cohort figures, and a wide table combining markers with an embedding.

## Set up a compact annotated object

The values are synthetic; the sample, condition, cell-type, marker, and PCA structure is representative of an ordinary visualization workflow.

In [ ]:
import anndata as ad
import annplyr as ap
import numpy as np
import pandas as pd
from scipy import sparse

adata = ad.AnnData(
    X=sparse.csr_matrix(
        np.array(
            [
                [8, 4, 1, 0],
                [6, 5, 0, 0],
                [0, 0, 7, 3],
                [0, 1, 5, 6],
                [4, 2, 0, 0],
                [1, 0, 2, 8],
            ],
            dtype=np.float32,
        )
    ),
    obs=pd.DataFrame(
        {
            "sample_id": ["S1", "S1", "S2", "S2", "S3", "S3"],
            "condition": ["control", "control", "stimulated", "stimulated", "control", "control"],
            "cell_type": ["B cell", "B cell", "T cell", "Monocyte", "B cell", "Monocyte"],
        },
        index=[f"cell_{i}" for i in range(6)],
    ),
    var=pd.DataFrame(index=["MS4A1", "CD79A", "NKG7", "LST1"]),
)
adata.obsm["X_pca"] = np.array(
    [[-1.0, 0.2], [-0.8, 0.1], [0.4, 1.0], [0.9, -0.2], [-0.4, 0.3], [0.7, -0.5]],
    dtype=np.float32,
)
adata

## Cell-level long data

Select only the genes represented in the figure. The result has stable `obs_name`, `feature`, and `value` columns plus the requested observation metadata.

In [ ]:
marker_long = adata.ap.to_tidy(
    obs=["sample_id", "condition", "cell_type"],
    x=["MS4A1", "CD79A", "NKG7"],
    max_matrix_values=3 * adata.n_obs,
)
marker_long.head(9)

A plotting library can consume `marker_long` directly without knowing about AnnData:

```python
import seaborn as sns

sns.catplot(
    data=marker_long,
    x="cell_type",
    y="value",
    hue="condition",
    col="feature",
    kind="strip",
    sharey=False,
)
```

Seaborn remains optional; annplyr owns only the extraction contract.

## Group-level data for cohort figures

When cells are not the inferential unit, summarize before extracting a large table. This creates one row per observed sample and cell type.

In [ ]:
sample_summary = adata.ap.summarize(
    obs={"cells": ap.n()},
    x={
        "mean_MS4A1": ap.mean("MS4A1"),
        "mean_NKG7": ap.mean("NKG7"),
    },
    by=["sample_id", "condition", "cell_type"],
    max_matrix_values=2 * adata.n_obs,
)
sample_summary

## A wide model table with PCA coordinates

`to_df()` keeps one row per cell. Keyed `obsm` selection makes embedding coordinates explicit and gives them collision-safe column names.

In [ ]:
model_frame = adata.ap.to_df(
    obs=["sample_id", "condition", "cell_type"],
    x=["MS4A1", "CD79A", "NKG7", "LST1"],
    obsm={"X_pca": ["0", "1"]},
    max_matrix_values=6 * adata.n_obs,
)
model_frame.head()

## Reshape after extraction

The pandas-only helpers are useful once data have crossed the extraction boundary. For example, a summary can be widened for a compact report table.

In [ ]:
report_table = ap.pivot_wider(
    sample_summary,
    id_cols=["sample_id", "condition"],
    names_from="cell_type",
    values_from="cells",
)
report_table

## Takeaway

Choose the smallest table that matches the downstream task: long selected-marker values for faceted cell plots, grouped summaries for cohort figures, or a wide selected-feature frame for modelling. Keeping feature selection and materialization budgets next to the extraction call makes that boundary reviewable.